# SIH26168 Colab Training — AVNetLite on IO-VNBD
Target 95% accuracy (<5% drift). Run all cells in order. T4 GPU recommended.

In [ ]:
!git clone https://github.com/Himanshu121865/sih26168.git
%cd sih26168
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q
!pip install pandas scipy onnx onnxruntime onnxscript loguru matplotlib scikit-learn -q
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
!python python/download_iovnbd.py --subset Sync
!ls -lh data/iovnbd/*.zip
# FULL Categorised 72 seqs (~1M windows, ~5GB npy, streaming via mmap) — ~7 mins preprocess, 35 mins train 50ep
!PYTHONPATH=. python python/preprocess.py --subset full --window 200 --stride 10 --hz 100
!ls -lh data/processed/ && cat python/scaler.json | head -20

In [ ]:
# Retrain with P0 fixes: ZUPT (5.7% stationary), random-yaw + NLL — 15ep is enough for full 822k
!PYTHONPATH=. python python/train_avnet.py --epochs 15 --batch 256 --lr 1e-3 --device cuda --augment-yaw --augment-bike --lambda-nll 0.1
!ls -lh experiments/checkpoints/

In [ ]:
!PYTHONPATH=. python python/eval_drift.py --model experiments/checkpoints/model_avnet_stage1.p --plot reports/drift_plot.png
from IPython.display import Image; Image("reports/drift_plot.png")

In [ ]:
!pip install ai-edge-torch -q
!PYTHONPATH=. python python/export_tflite.py --model experiments/checkpoints/model_avnet_stage1.p --out model.tflite --onnx model.onnx
!ls -lh model.* scaler.json
!zip -r screening.zip model.tflite scaler.json reports/drift_plot.png reports/drift_2d.png python/scaler.json
!ls -lh screening.zip


In [ ]:
# F1-F7 verification (added 2026-09-04)
!PYTHONPATH=. python python/inekf_harness.py --test-lean
!PYTHONPATH=. python python/inekf_harness.py --model experiments/checkpoints/model_avnet_stage1.p --windows 600 --start 2000 --lean-mode car --q-acc 30.0
!PYTHONPATH=. python python/eval_drift.py --mode 2d --model experiments/checkpoints/model_avnet_stage1.p --plot reports/drift_2d.png
from IPython.display import Image; Image("reports/drift_2d.png")


In [ ]:
# Option A audit (per-file val) — run after train
!PYTHONPATH=. python python/eval_per_file.py --model experiments/checkpoints/model_avnet_stage1.p


In [ ]:
# python-pro validation: types + tests (pure modules)
!pip install pytest pytest-cov -q
!PYTHONPATH=. python -m pytest tests/ -q
